# B0 -- model viability (steps B0-3, B0-4, B0-5, B0-6)

Branch `milestone1/b0-model-viability`. Exploratory; the durable outputs are the constants in
`boatphone/config.py`, the CPU shim in `boatphone/onc_model_cpu.py`, and the findings reported
to the orchestrator.

**Everything here is measured against the pinned artefacts**, not read off a README.
Provenance: `docs/derived/b0_external_provenance.json`.

**Third-party packages are NOT in the shared hub environment.** `timm`, `einops`, `mamba_ssm`,
`triton` and `transformers` were installed OUT OF TREE with
`pip install --no-deps --target <dir>` and put on `PYTHONPATH`. Nothing was installed into
`/home/.pixi/envs/default`. Set `EXTRA_LIBS` below to that directory before running the model
cells.


In [ ]:
import os, sys, pickle, time, json, collections
import numpy as np

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

# Out-of-tree install directory for timm / einops / mamba_ssm / triton / transformers.
# None -> the model cells will raise with install instructions rather than guess.
EXTRA_LIBS = os.environ.get("BOATPHONE_EXTRA_LIBS")
if EXTRA_LIBS:
    sys.path.insert(0, EXTRA_LIBS)

from boatphone import paths, config

CKPT_DIR = paths.require_path(paths.CHECKPOINT_DIR)
ARGS_PKL = paths.require_path(CKPT_DIR / "finetune" / "args.pkl")
CKPT_PTH = paths.require_path(CKPT_DIR / "finetune" / "ft-cls_best_checkpoint.pth")
EVAL_H5  = paths.require_path(
    CKPT_DIR / "eval" / "different_locations_incl_backgroundpipelinenormals_multilabel_SMALL.h5"
)
ONC_REPO = paths.require_path(paths.ONC_MODEL_DIR)
print("all pinned inputs present")


## B0-5 -- dependency audit

Resolve every import the eval path, the model definition and `utilities/spectrogram_utils.py`
need, against **this** environment. `cnn_baseline/` does not exist (confirmed in B0-1 and again
below), so there is nothing to audit there.


In [ ]:
import importlib, importlib.metadata as md

REQUIRED = [
    # (module, why it is needed)
    ("numpy",        "everywhere"),
    ("scipy",        "spectrogram_utils.load_mat_spectrogram -> scipy.io.loadmat"),
    ("h5py",         "onc_ssamba/dataset.py"),
    ("cv2",          "spectrogram_utils.resize_to_target"),
    ("torch",        "everything"),
    ("torchaudio",   "requirements.txt; not actually imported on the eval path"),
    ("sklearn",      "utilities/metrics/*"),
    ("pandas",       "eval/evaluate_model.py"),
    ("matplotlib",   "models/both_models.py imports pyplot AT MODULE LEVEL"),
    ("tqdm",         "optional, guarded fallback in the repo"),
    ("timm",         "models/both_models.py + models_mamba.py, UNGUARDED"),
    ("einops",       "models_mamba.py / mamba_ssm"),
    ("mamba_ssm",    "models_mamba.py, UNGUARDED at module import time"),
    ("causal_conv1d","optional in mamba_ssm; absent -> F.conv1d fallback"),
    ("triton",       "pulled in by mamba_ssm.ops.triton.layer_norm"),
    ("transformers", "pulled in by mamba_ssm.utils.generation"),
    ("wandb",        "traintest.py, training only"),
]

rows = []
for name, why in REQUIRED:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", None) or md.version(name)
        where = "hub env" if "/.pixi/" in (getattr(mod, "__file__", "") or "") else "out-of-tree"
        rows.append((name, "PRESENT", ver, where, why))
    except ImportError:
        rows.append((name, "ABSENT", "-", "-", why))

for r in rows:
    print(f"{r[0]:<14} {r[1]:<8} {r[2]:<10} {r[3]:<13} {r[4]}")


In [ ]:
import torch
print("torch", torch.__version__, "| torch.cuda.is_available() =", torch.cuda.is_available())

# CLAUDE.md's "not available" list is STALE on cv2: it IS present.
import cv2
print("cv2", cv2.__version__, "-- CLAUDE.md lists cv2-adjacent tooling as unavailable; wrong here")

# cnn_baseline/ really is absent from the clone.
print("cnn_baseline present:", (paths.ONC_MODEL_DIR / "cnn_baseline").exists())


In [ ]:
# Does the model definition import mamba_ssm unconditionally, or only on the SSL path?
src = (paths.ONC_MODEL_DIR / "onc_ssamba" / "models" / "models_mamba.py").read_text()
head = src.split("__all__")[0]
for line in head.splitlines():
    if "mamba_ssm" in line or "timm" in line:
        print(repr(line))
# -> the `from mamba_ssm.modules.mamba_simple import Mamba` line is at module top level and is
#    NOT inside a try/except. Importing onc_ssamba.models fails outright without mamba_ssm.


## B0-4 -- class ordering

Two sources were asked for. Only one of them exists.

* `args.pkl` carries **no ordered label list at all**. It carries `n_class = 2` /
  `num_classes = 2`, which describe the *unused* VisionMamba `v.head`.
* the eval h5 has **no `label_names` dataset**. Its datasets are `index_map_original`,
  `label_strings`, `labels`, `sources`, `spectrograms`, `split/*`.

So the ordering is recovered from the data itself: cross-tabulate each column of `labels`
against `label_strings`. The YAML's order is never taken; it is only used afterwards as a check.


In [ ]:
a = pickle.load(open(ARGS_PKL, "rb"))
argsd = vars(a) if hasattr(a, "__dict__") else a
print("args.pkl keys containing 'class' or 'label':",
      sorted(k for k in argsd if "class" in k or "label" in k))
print({k: argsd[k] for k in ("n_class", "num_classes", "exclude_labels", "task", "loss")})
print("ordered label list in args.pkl?", any(isinstance(v, (list, tuple)) and v and
      isinstance(v[0], str) for v in argsd.values()))


In [ ]:
import h5py
with h5py.File(EVAL_H5, "r") as hf:
    print("datasets:", list(hf.keys()), "| 'label_names' present:", "label_names" in hf)
    L  = hf["labels"][:]
    LS = [s.decode() for s in hf["label_strings"][:]]
print("labels", L.shape, "column sums", L.sum(0).tolist())

# Recover the ordering: for each column, which label token explains every row that has it set?
recovered = []
for c in range(L.shape[1]):
    rows_c = np.where(L[:, c] == 1)[0]
    tokens = [set(t.strip(" b'\"") for t in LS[i].strip("b'\"").split(";")) for i in rows_c]
    common = set.intersection(*tokens) if tokens else set()
    # the explaining token is the one shared by every row of this column and by no other column
    recovered.append(sorted(common))
for c, names in enumerate(recovered):
    print(c, names)


In [ ]:
# Compare against the YAML -- as a CHECK, never as the source.
yaml_txt = (paths.ONC_MODEL_DIR / "config" / "dataset_config.yaml").read_text()
block = yaml_txt.split("anomaly_labels:", 1)[1]
yaml_order = []
for line in block.splitlines()[1:]:
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        continue
    if not stripped.startswith('- "'):
        break                      # end of the list
    yaml_order.append(stripped.split('"')[1])
print("YAML order  :", yaml_order)
print("data-derived:", list(config.ONC_MODEL_LABEL_NAMES))
print("agree:", yaml_order == list(config.ONC_MODEL_LABEL_NAMES))
print()
print("Engine Noise index in the DATASET label matrix:", config.ENGINE_NOISE_LABEL_INDEX)
print("logits the model actually emits:", config.ONC_MODEL_N_LOGITS)


In [ ]:
# The decisive check for what 'Engine Noise index' can mean: read the head shape off the
# checkpoint. mlp_head.1.weight is (1, 768) -> ONE logit -> binary normal-vs-anomalous.
ck = torch.load(CKPT_PTH, map_location="cpu", mmap=True)   # weights_only default does NOT bite
sd = ck["model_state_dict"]
for k in ("v.head.weight", "mlp_head.1.weight", "v.pos_embed", "v.patch_embed.proj.weight"):
    print(f"{k:<28} {tuple(sd[k].shape)}")
print("checkpoint epoch:", ck["epoch"], "| val_acc:", ck["val_acc"])


## B0-3 -- the reproduction target

`result.csv` is **not in the clone**: `traintest.py` writes it to `args.exp_dir`, and `exp/` is
gitignored. It does exist on the Hub, in `merileo/onc-ssl-tutorial` under the *same experiment
directory name* as the checkpoint we pinned. Fetch it and read the `ICLISTENHF1266` columns.


In [ ]:
REPRO = CKPT_DIR / "repro" / (
    "finetune/amba-base-f16-t16-b16-lr1e-4-m300-custom-tr0.8-full_dataset_hydrophones-noexclude"
)
result_csv = paths.require_path(REPRO / "result.csv")
import csv
rows = [r for r in csv.reader(open(result_csv)) if r]
hdr, data = rows[0], rows[1:]
devs = [h[:-len("_count")] for h in hdr if h.endswith("_count")]
base = hdr.index(devs[0] + "_count")
FIELDS = ["count", "predictions", "targets", "precision", "recall", "f2", "auc"]
print("epochs:", len(data), "| per-device blocks:", len(devs), "| fields per block:", FIELDS)

def block(row, i):
    return dict(zip(FIELDS, row[base + 7*i: base + 7*i + 7]))

i66 = devs.index("ICLISTENHF1266")
print("\nEPOCH 1, columns named ICLISTENHF1266_* -- verbatim:")
print(block(data[0], i66))


In [ ]:
# Is the header->device mapping stable across epochs? `hydrophone_metrics` is a defaultdict
# keyed in first-appearance order over the validation loader; the header is written ONCE, at
# epoch 1. Track the block whose `targets` == 25 (HF1266's positive count at epoch 1).
print("epoch | header name of the block with targets==25 | count  P      R      AUC")
for r in data:
    for i, d in enumerate(devs):
        b = block(r, i)
        if float(b["targets"]) == 25.0:
            print(f"{r[0]:>5} | {d:<26} | {b['count']:>5}  "
                  f"{float(b['precision']):.4f} {float(b['recall']):.4f} {float(b['auc']):.4f}")
# The name moves. From roughly epoch 3 onward a row cannot be attributed to a device by its
# header name, and the per-device `count` moves too (149 -> 139 -> 148), so the validation
# subset is not fixed either. n=149 is an EPOCH-1 figure, not a fixed test-set size.


In [ ]:
# Is the checkpoint we pinned the one behind this result.csv?
from huggingface_hub import HfApi
api = HfApi()
info = api.repo_info("merileo/onc-ssl-tutorial", files_metadata=True)
want = str(REPRO.name) + "/models/ft-cls_best_checkpoint.pth"
for s in info.siblings:
    if s.rfilename.endswith(want) and s.lfs:
        print("tutorial-repo checkpoint sha256:", s.lfs["sha256"])
prov = json.loads((paths.DOCS_DIR / "derived" / "b0_external_provenance.json").read_text())
pinned = prov["merileo_checkpoint_bundle"]["sha256"]["external/checkpoints/finetune/ft-cls_best_checkpoint.pth"]
print("pinned checkpoint sha256:      ", pinned)


In [ ]:
# Metric SEMANTICS -- these matter more than the numbers.
print("threshold          :", config.ONC_MODEL_DECISION_THRESHOLD,
      "(hydrophone_metrics.calculate_binary_metrics default)")
print("split              : VALIDATION (traintest.py writes result.csv per validation epoch)")
print("split ratios       :", {k: argsd[k] for k in ("train_ratio", "val_ratio", "split_seed")})
print("positive class     : is_anomalous == ANY label other than 'normal'")
print("                     -> NOT Engine Noise, and NOT macro-averaged; it is one binary class")
print("checkpoint epoch   :", ck["epoch"], "(last epoch; best_metrics in the file was never updated:",
      ck["best_metrics"], ")")
print("n                  : SAMPLES (one 512x512 spectrogram window per row of the h5)")


## B0-6 -- does the checkpoint load and run a forward pass on CPU?

`torch.load` succeeds with the **default** `weights_only` (the file holds only tensors and plain
containers), given `mmap=True` -- without mmap the 1.1 GB read was OOM-killed once on this host.

The model *definition* cannot be imported at all without `mamba_ssm`, and `mamba_ssm==2.2.5`
does an unguarded `import selective_scan_cuda`. `boatphone/onc_model_cpu.py` is the one
documented workaround: stub the CUDA extension so every entry point raises, substitute
`mamba_ssm`'s own pure-PyTorch `selective_scan_ref`, and force every mixer off the fast path.

**Set `RUN_FORWARD_PASS = True` only if you want to reproduce the failure.** On this host
(30 GB RAM, 4 cores, no CUDA) the forward pass is OOM-killed -- it kills the kernel, so it is
off by default. Measured 2026-08-27, see the markdown after the cell.


In [ ]:
RUN_FORWARD_PASS = False

from boatphone import onc_model_cpu as shim

if RUN_FORWARD_PASS:
    print(shim.prepare_cpu_mamba())
    shim.add_onc_repo_to_path(paths.ONC_MODEL_DIR)
    from onc_ssamba.models import AMBAModel

    d = argsd
    vmc = dict(
        img_size=(d["num_mel_bins"], d["target_length"]), patch_size=d["patch_size"],
        stride=d["stride"], embed_dim=d["embed_dim"], depth=d["depth"], channels=d["channels"],
        num_classes=d["num_classes"], drop_rate=d["drop_rate"], drop_path_rate=d["drop_path_rate"],
        norm_epsilon=d["norm_epsilon"], rms_norm=d["rms_norm"],
        residual_in_fp32=d["residual_in_fp32"], fused_add_norm=d["fused_add_norm"],
        if_rope=d["if_rope"], if_rope_residual=d["if_rope_residual"],
        bimamba_type=d["bimamba_type"], if_cls_token=d["if_cls_token"],
        # NOTE the typo in the checkpoint's args: `if_devide_out`, not `if_divide_out`.
        # create_model() reads `args.if_divide_out` and would AttributeError on this args.pkl.
        if_divide_out=d["if_devide_out"],
        use_double_cls_token=d["use_double_cls_token"],
        use_middle_cls_token=d["use_middle_cls_token"],
        if_bidirectional=d["if_bidirectional"], final_pool_type=d["final_pool_type"],
        if_abs_pos_embed=d["if_abs_pos_embed"], if_bimamba=d["if_bimamba"],
    )
    label_dim = 1 if (d["task"] == "ft_cls" and d["n_class"] == 2) else d["n_class"]

    model = AMBAModel(
        label_dim=label_dim, fshape=d["fshape"], tshape=d["tshape"], fstride=d["fstride"],
        tstride=d["tstride"], input_fdim=d["num_mel_bins"], input_tdim=d["target_length"],
        model_size=d["model_size"], pretrain_stage=False, load_pretrained_mdl_path=None,
        vision_mamba_config=vmc, allow_random_init_finetune=True,
    )

    # REPO/CHECKPOINT DRIFT, restored explicitly. The pinned repo revision hard-codes
    # VisionMamba num_classes = label_dim (= 1); the checkpoint's v.head is (2, 768) because it
    # was trained with args.num_classes = 2. v.head is UNUSED by finetuningcls. strict=True must
    # stay strict, so rebuild v.head to the shape the checkpoint itself declares.
    want = sd["v.head.weight"].shape
    if tuple(model.v.head.weight.shape) != tuple(want):
        print("DRIFT: repo builds v.head", tuple(model.v.head.weight.shape), "checkpoint has", tuple(want))
        model.v.head = torch.nn.Linear(want[1], want[0])

    res = model.load_state_dict(sd, strict=True)
    print("load_state_dict(strict=True) -> missing", res.missing_keys, "unexpected", res.unexpected_keys)
    print("forced onto slow path:", shim.force_slow_path(model), "mixers")
    model.eval()
else:
    print("skipped -- see the recorded measurement below")


In [ ]:
# One REAL spectrogram from the eval h5, through the repo's OWN preprocessing.
# No resizing, no truncation, no shape coercion: the h5 rows are already (512, 512, 1) and
# AMBAModel.forward raises for anything else.
with h5py.File(EVAL_H5, "r") as hf:
    srcs = [s.decode() for s in hf["sources"][:]]
    # NOTE: `sources` in this h5 are DOUBLE-ENCODED -- the stored string literally begins
    # with  b'  . The repo's extract_hydrophone() splits on '_' and would yield "b'ICLISTENHF1266".
    idx_66 = [i for i, s in enumerate(srcs) if config.DEVICE_CODE in s]
    labels_66 = hf["labels"][:][idx_66]
    print("HF1266 rows in the SMALL eval h5:", len(idx_66))
    for name, n in zip(config.ONC_MODEL_LABEL_NAMES, labels_66.sum(0)):
        print(f"   {name:<16} {int(n)}")
    idx = idx_66[0]
    raw = hf["spectrograms"][idx]
print("chosen row", idx, srcs[idx], raw.shape, raw.dtype)

x = raw.astype(np.float32)
x = (x - config.ONC_MODEL_DATASET_MEAN) / (config.ONC_MODEL_DATASET_STD * 2)   # dataset.normalise
x = np.nan_to_num(x, 0)
t = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0)   # [B, C, F, T]
print("input tensor", tuple(t.shape))


In [ ]:
if RUN_FORWARD_PASS:
    outs = []
    for r in range(2):
        t0 = time.time()
        with torch.no_grad():
            o = model(t, task=argsd["task"])
        print(f"run {r}: {time.time()-t0:.2f} s  logits={o.numpy().ravel().tolist()}")
        outs.append(o.detach().clone())
    print("bitwise deterministic across two runs:", torch.equal(outs[0], outs[1]))
    print("output vector length:", outs[0].numel(), "vs label count in the h5: 8")
else:
    print("skipped")


### B0-6 result, measured 2026-08-27 on the OHW hub (4 cores, 30 GB RAM, no CUDA)

| Question | Answer |
|---|---|
| `torch.load` under torch 2.12 | **OK.** Default `weights_only` does *not* bite. `mmap=True` is required in practice -- a plain read of the 1.1 GB file was OOM-killed once. |
| model definition importable on CPU | **Only via `boatphone/onc_model_cpu.py`.** `mamba_ssm` is imported unconditionally by `models_mamba.py`, and `mamba_ssm==2.2.5` does an unguarded `import selective_scan_cuda`. |
| weights load | **Clean.** `load_state_dict(strict=True)` -> `missing []`, `unexpected []`, after restoring `v.head` to the (2, 768) the checkpoint declares. 276 tensors, 92.66 M parameters. |
| forward pass | **NO. Killed by the OS (exit 137).** |
| logit vector length | not reached; the checkpoint's `mlp_head.1.weight` is (1, 768), so it would be **1**, against 8 dataset labels. |
| Engine Noise read-out | **not available at any index** -- one binary logit, see B0-4. |
| determinism | not reached. |
| seconds/sample on CPU | **not measurable.** |
| projected cost of scoring the ~2.7 GB corpus | **cannot be projected.** |

The failure is sharp and reproducible. Isolating one `Mamba(d_model=768, d_state=16)` layer and
sweeping the sequence length, with `mamba_ssm`'s reference selective scan:

| sequence length | wall clock | peak RSS |
|---|---|---|
| 128 | 0.20 s | 0.69 GB |
| 512 | 0.50 s | 0.81 GB |
| 1024 | 1.52 s | 0.97 GB |
| 2048 | 2.26 s | 1.17 GB |
| 2200 | 2.38 s | 1.18 GB |
| 2400 | -- | **OOM, exit 137** |
| 2501 (**the length this model needs**) | -- | **OOM, exit 137** |

2501 = 50x50 patches + 1 cls token, fixed by `v.pos_embed` being (1, 2501, 768).
Also reproduced with `OMP_NUM_THREADS=1`, so it is not a thread-buffer artefact. The individual
allocation was not isolated: the two `einsum`s and the depthwise `conv1d` inside
`selective_scan_ref` were each timed at L=2400 and L=2501 and all stayed under 1 GB. The
30-minute workaround timebox was spent; the honest answer is *the method is broken on this host*,
not *the method found nothing*.


## Null check

There is no correlation to null-test yet -- no forward pass ran, so no score exists to correlate
with anything. The check that *was* run is the structural analogue: the `result.csv`
header-to-device attribution was tested against a signature (`targets == 25`) instead of trusted,
and it **failed** -- the name attached to that block changes from roughly epoch 3 onward. Had the
header been believed, every per-device number after epoch 2 would have been attributed to the
wrong hydrophone, which is exactly the class of silent-corruption error decision 0002 exists to
prevent.
